# Issue #51 — Full S20 resolution training

Two independent 20-epoch YOLO26n arms, one fixed Raw checkpoint-selection procedure, and four development evaluations. This is single-seed development evidence; Issue #47 remains open.

**Inputs:** Finalized `docs/ISSUE47_FULL_TRAINING_PLAN.md` and verified implementation receipt. Acquisition additionally requires the frozen S20/development metadata, existing verified v0 Raw JPEG95 exports, pinned YOLO26n and CUDA.

**Outputs:** New owned `runs/issue47-s20-resolution-20e/` and compact `docs/issue47-s20-resolution-20e/` evidence. No historical artifact is overwritten.

**Prerequisites:** Commit the verified protocol/implementation before explicitly enabling acquisition. No upstream notebook runs. Confirmation and final pixels remain sealed. This implementation has not acquired real data or trained a GPU model.

## 1. Configuration

All acquisition flags default to False. Restart & Run All verifies frozen code/plan and saved compact evidence, or reports not acquired, without loading datasets, annotations, models or local runs. Enable only the stage explicitly intended; stages read saved prerequisites.

In [ ]:
from pathlib import Path

import pandas as pd
from IPython.display import Image, display

from vehicle_detection.common import require_paths
from vehicle_detection.full_resolution_training import (
    PLAN,
    RECEIPT,
    STUDY,
    evaluate,
    prepare,
    read_report,
    train,
)

ROOT = next(p for p in [Path.cwd(), *Path.cwd().parents] if (p / "pyproject.toml").is_file())
require_paths(protocol=ROOT / PLAN, verification=ROOT / RECEIPT)
ALLOW_PREPARATION = False
ALLOW_TRAINING = False
ALLOW_INFERENCE = False
result = read_report(ROOT)
print(result["status"])

## 2. Verify and prepare owned Raw data

Explicit preparation verifies all 9,781 training / 13,975 development IDs, group separation and existing Raw export receipts. It copies data once, checks label/image bytes and geometry, computes fixed development cohorts, and records historical preservation. Existing or partial attempts block preparation. No original source pixels or confirmation/final pixels are needed.

In [ ]:
if ALLOW_PREPARATION:
    prepare(ROOT, allow_preparation=True)

## 3. Two native training arms with fixed-640 selection

Same pinned initialization, seed 2026, batch 16, FP32 AdamW, frozen backbone 11 and disabled augmentations; only training resolution differs. Both train for 20 epochs, sequentially, without automatic recovery or resume.

Selection uses the same Raw development at actual maximum tensor edge 640, native rectangular dataset with pad=0, batch 32, native fitness/best.pt and every-epoch plus terminal validation. Training remains square 640/960. Saved pre-strip checkpoint receipts identify selected epochs, including ties. Native CPU tests verify this behavior before real acquisition.

In [ ]:
if ALLOW_TRAINING:
    train(ROOT, allow_training=True)

## 4. Fixed matrix and saved evidence

Evaluate the same selected checkpoint for each arm at both inference sizes, in fixed order A (640/640), D (960/960), B (640/960), C (960/640). Inference keeps native rectangular pad=.5; record actual shapes. Primary D−B isolates training resolution at inference 960. The 1,000 paired original-source-group draws for all/dark/bright are CPU analysis only, fixed before outcomes.

In [ ]:
if ALLOW_INFERENCE:
    evaluate(ROOT, allow_inference=True)
result = read_report(ROOT)
if result["status"] == "completed":
    display(
        pd.DataFrame(
            [
                {
                    "train/infer": key,
                    "mAP50:95 (%)": 100 * m["all"]["mAP50_95"],
                    "mAP50 (%)": 100 * m["all"]["mAP50"],
                }
                for key, m in result["metrics"].items()
            ]
        )
    )
    display(Image(filename=str(ROOT / f"docs/{STUDY}/convergence_matrix.png")))
    display({key: 100 * value for key, value in result["contrasts"].items()})
else:
    print(result["status"], "— no completed full-S20 results exist.")

## 5. Class support, errors and uncertainty

Report motorcycle/bicycle and dark-bus GT counts, source groups and individual intervals. Sparse support, intervals crossing zero or AP gains with worse fixed-confidence TP/FN cannot become a clean success. Counts use conf≥.25 / IoU .50 from native matches; AP uses conf=.001. Cohort AP retains all GT and is not AP-small. Intervals are conditional on selected models and reused development, without training-seed uncertainty or multiplicity correction.

In [ ]:
if result["status"] == "completed":
    for key, metrics in result["metrics"].items():
        for cohort in ("all", "dark", "bright", "small_two_wheeler"):
            print(key, cohort)
            display(pd.DataFrame(metrics[cohort]["classes"]).T)
            display(pd.DataFrame(metrics[cohort]["counts"]).T)
    for cohort, interval in result["cohort_intervals"].items():
        print(cohort)
        if interval is not None:
            print(
                "Draws / valid / missing-class:",
                interval["resamples"],
                interval["valid_draws"],
                interval["invalid_draws"],
            )
            display(pd.DataFrame(interval["per_class"]))
    display(pd.DataFrame(result["examples"]))

## 6. Convergence, cost and Summary

Inspect all 20 epochs, train/val losses, learning rates, selection peaks and final trends. Do not extend the budget based on curves. The ~3 GPU-hours is a planning ceiling, not measured runtime. Abort limits are 90/150 minutes per arm and 14 GiB allocated VRAM. GPU batch p95 is not per-image p95 or deployment latency. Cross-version S20 comparisons stay in experiments/compare.ipynb.

In [ ]:
if result["status"] == "completed":
    for size in (640, 960):
        print("Training", size)
        display(pd.read_csv(ROOT / f"docs/{STUDY}/train{size}_results.csv"))
    display(pd.DataFrame(result["training"]).T)
    display(pd.DataFrame(result["timings"]).T)
    display(result["decision"])
    assert not result["confirmation_final_accessed"]
    assert not result["decision"]["policy_selected"]
    assert not result["decision"]["independent_confirmation"]
print("Issue #47 remains open; this study does not release an adaptive policy.")